<a href="https://colab.research.google.com/github/RPG-coder/newswise-bbc-kaggle/blob/main/Amazon_Reviews_2023_small.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install kaggle
%pip install langchain langchain-core langchain-community langchain-anthropic langchain-huggingface langgraph faiss-gpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1
ERROR: pip's dependency re

In [2]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

In [3]:
!kaggle datasets download danielrosel/mcauley-labamazon-reviews-2023-small
!unzip mcauley-labamazon-reviews-2023-small.zip

Dataset URL: https://www.kaggle.com/datasets/danielrosel/mcauley-labamazon-reviews-2023-small
License(s): unknown
100% 51.0M/51.0M [00:03<00:00, 13.4MB/s]

Archive:  mcauley-labamazon-reviews-2023-small.zip
  inflating: df_final.csv            


In [4]:
import pandas as pd
original_df = pd.read_csv("df_final.csv")

In [5]:
original_df.shape # (208965, 10): The dataset has a lot of row. for testing lets sample till 5000 first
original_df['description'].isna().sum() # Is the dataset clean? np.int64(0)
original_df.columns.tolist() # what are the columns in the dataset?

['Unnamed: 0',
 'rating',
 'images',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase',
 'description']

In [6]:
original_df.head() # show me first 5 row in dataset.

,Unnamed: 0,rating,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,description
0,1777566,3.0,"[{'attachment_type': 'IMAGE', 'large_image_url...",B073VDZ9SX,B073VDZ9SX,AG2YH7YX4G3IEW6CC4RCEAXDEANA,2022-12-13 21:31:22.378,0,True,Could never get it to attach smoothly. While t...
1,11206598,1.0,"[{'attachment_type': 'IMAGE', 'large_image_url...",B008NNVIIE,B008NNVIIE,AFN4DLC3C32GRBPKF4R67NQXN4MA,2022-09-17 22:12:07.387,0,True,Seriously! Came only with 3 containers
2,3731411,4.0,"[{'attachment_type': 'IMAGE', 'large_image_url...",B08WTD9WH2,B08WTD9WH2,AFECIF57BBZ6AR4HVP3H6BWI5AOQ,2021-06-02 19:36:14.611,7,True,Doesn't fit 11x17 paper without folding it in ...
3,9135045,5.0,"[{'attachment_type': 'IMAGE', 'large_image_url...",B07R6LKZS9,B07R6LKZS9,AGNFE4NTSVVRJMW6OBE7LGDYFP6A,2019-08-08 19:08:29.563,0,True,MUST HAVE FOR STUDENTS/BUSINESS OWNERS I love ...
4,9197907,5.0,"[{'attachment_type': 'IMAGE', 'large_image_url...",B07V51W2K6,B084VV4KKF,AFJ4N5SBODFS2X5TNL2GFICFJRHQ,2021-03-23 05:29:49.853,0,True,Buy it you’ll love it!! Super cute!! Love it! ...


In [7]:
df = original_df.sample(n=5000, random_state=42) # lets sample 5000 row
df.shape

(5000, 10)

In [14]:
from langchain_core.documents import Document
docs = df.apply(
    lambda x: Document(
        page_content=f"{x['description']}",
        metadata={
          'asin': x['asin'],
          'parent_asin': x['parent_asin'],
          'reviewerID': x['user_id'],
          'rating': x['rating'],
          'reviewTime': x['timestamp'],
          'helpful_vote': x['helpful_vote'],
          'verified_purchase': x['verified_purchase'],
        }
    ),
    axis=1,
).tolist()

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(doc, embedding)

/tmp/ipykernel_783/2354072734.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [19]:
def format_doc(docs):
  return "\n\n".join(
      f"""Page Content: {doc.page_content}
Ratings: {doc.metadata['rating']}
Review Timestamp: {doc.metadata['reviewTime']}
Total Helpful vote: {doc.metadata['helpful_vote']}
Verified purchase: {'Yes' if doc.metadata['verified_purchase'] else 'No'}"""
      for doc in docs
  )

In [21]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

history = []
prompt = ChatPromptTemplate.from_messages(
    messages=[
        ('system', "You are an helpful assistant. Answer with provided context. If you are not sure, say you don't know."),
        MessagesPlaceholder(variable_name="history"),
        ('human', "Question: {question}\nContext: {context}\nResponse:"),
    ]
)

In [22]:
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-sonnet-4-6")

In [24]:
from typing import TypedDict

class GraphState(TypedDict):
  question: str
  answer: str

In [48]:
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import HumanMessage, AIMessage

def ask_question(state: GraphState) -> GraphState:
  return {
      'question': input()
  }

def generate_answer(state: GraphState) -> GraphState:
  chain = {
      'question': RunnableLambda(lambda x: x['question']),
      'context':  RunnableLambda(lambda x: x['question']) | retriever | format_doc,
      'history': RunnableLambda(lambda x: x['history']),
  } | prompt | llm
  response = chain.invoke({
      'question': state['question'],
      'history': history,
  })
  print('Agent: ', response.content)
  history.append(HumanMessage(content=state['question']))
  history.append(AIMessage(content=response.content))
  return {
      'answer': response.content
  }

In [49]:
from langgraph.graph import StateGraph, START, END
builder = StateGraph(GraphState)

def route(state: GraphState) -> str:
    return END if state['question'] == '/exit' else 'answer'

In [50]:
builder.add_node('ask', ask_question)
builder.add_node('answer', generate_answer)
builder.add_edge(START, 'ask')
builder.add_conditional_edges('ask', route)
builder.add_edge('answer', 'ask')

In [51]:
from langgraph.checkpoint.memory import MemorySaver
saver = MemorySaver()
graph = builder.compile(checkpointer=saver)

In [52]:
config = {"configurable": {"thread_id": "amazonReview-1"}}
graph.invoke({
    'question': '',
    'answer': '',
}, config=config)

"Are there any products with poor quality?"
Agent:  ## Answer

**Yes**, , there is at least one product with poor quality based on the reviews provided.

### Poor Quality Review Details:
| Field | Details |
|-------|---------|
| **Review** | Poor quality |
| **Rating** | ⭐ 1.0 out of 5.0 |
| **Review Date** | January 14, 2023 |
| **Helpful Votes** | 1 |
| **Verified Purchase** | ✅ Yes |

### Summary:
- **1 out of 3** reviews explicitly mentions **poor quality** with a rating of **1.0**
- The other **2 reviews** are positive, rating the products **5.0**, describing them as:
  - *"Good product, reasonable price"*
  - *"Good Quality Product at a fair price!"*

So while the **majority of reviews are positive**, there is **one verified purchase** that reported poor quality.
"What do verified buyers say about this product?"
Agent:  ## What Verified Buyers Say

All **3 reviews** are from **verified buyers** and all rate the product **5.0 stars** ⭐⭐⭐⭐⭐

---

### Review Highlights:

#### Review

{'question': '/exit',
 'answer': '## Is This Product Worth the Price?\n\n**Yes!** Based on verified buyer reviews, this product is **absolutely worth the price!** 💯\n\n---\n\n### Review Breakdown:\n\n#### Review 1 ✅\n> *"Good product, reasonable price"*\n- **Rating:** ⭐ 5.0\n- **Date:** January 4, 2022\n- **Verdict:** 👍 Reasonably priced\n\n---\n\n#### Review 2 ✅\n> *"Good Quality Product at a fair price!"*\n- **Rating:** ⭐ 5.0\n- **Date:** September 1, 2018\n- **Helpful Votes:** 1\n- **Verdict:** 👍 Fair price for good quality\n\n---\n\n#### Review 3 ✅\n> *"Worth every penny, great value...So when I saw this for a couple bucks I knew it would be a good find. Definitely improved my coloring. Love it"*\n- **Rating:** ⭐ 5.0\n- **Date:** October 17, 2019\n- **Verdict:** 👍 Worth every penny!\n\n---\n\n### Overall Summary:\n| Metric | Result |\n|--------|--------|\n| **Average Rating** | ⭐ 5.0/5.0 |\n| **Verified Purchases** | ✅ All 3 |\n| **Value Mentions** | 3 out of 3 |\n| **Recommended**